# IKG Column Lineage Explorer

Two modes:
- **Column / Table** — pick column + profile table → single graph HTML
- **Insight Type** — pick ODM insight → one graph section per rule_column HTML

Run cells 1–5, then run Mode 1 and/or Mode 2.


## 1. Imports

In [ ]:
import pandas as pd
import json
import os
import re
import datetime
from collections import deque
from IPython.display import display as ipy_display, clear_output, HTML as IPHTML
import ipywidgets as widgets


## 2. Configuration

In [ ]:
GP_HOST = 'greenplum-rdsp.zur.swissbank.com'
GP_PORT = 5432
GP_DB   = 'gprdsp'
GP_USER = 'ds_rdsp_dev'
GP_PASSWORD = ''   # leave blank — will prompt

IKG_SCHEMA = 'sandbox_prj_smart_insights'
ODM_SCHEMA = 'core_ikg'
LIN_TABLE  = 'ikg_column_lineage_master_auto_refresh'
ODM_TABLE  = 'odm_rule_metadata_auto_refresh'


## 3. Connect

In [ ]:
import getpass, sqlalchemy
if not GP_PASSWORD:
    GP_PASSWORD = getpass.getpass('Greenplum password: ')
engine = sqlalchemy.create_engine(
    f'postgresql+psycopg2://{GP_USER}:{GP_PASSWORD}@{GP_HOST}:{GP_PORT}/{GP_DB}'
)
print('Connected')


## 4. Load Data

In [ ]:
df_lin = pd.read_sql(f'SELECT * FROM {IKG_SCHEMA}.{LIN_TABLE}', engine).fillna('')
print(f'Lineage: {len(df_lin):,} rows')

def is_profile(r):
    tt  = str(r.get('target_table',     '') or '').lower().strip()
    stt = str(r.get('sub_target_table', '') or '').lower().strip()
    return tt == stt and tt.endswith('_profile_curr_ikg')

_profile_mask = df_lin.apply(is_profile, axis=1)
df_profile    = df_lin[_profile_mask]

COLS_LIST = sorted([
    c for c in df_profile['target_column'].dropna().unique().tolist()
    if c and str(c).strip()
])
print(f'Profile columns: {len(COLS_LIST)}')

try:
    df_it   = pd.read_sql(
        f'SELECT DISTINCT insight_type FROM {ODM_SCHEMA}.{ODM_TABLE}'
        ' WHERE insight_type IS NOT NULL ORDER BY insight_type', engine)
    IT_LIST = [it for it in df_it['insight_type'].tolist() if it and str(it).strip()]
    print(f'Insight types: {len(IT_LIST)}')
except Exception as e:
    IT_LIST = []
    print(f'Insight types unavailable: {e}')


## 5. Helpers

In [ ]:
_rows = df_lin.to_dict('records')

def _get(tbl, col):
    t, c = tbl.lower(), col.lower()
    return [r for r in _rows
            if str(r.get('sub_target_table','') or '').lower() == t
            and str(r.get('target_column',   '') or '').lower() == c]

def _schema(tbl):
    t = tbl.lower()
    for r in _rows:
        if str(r.get('source_table','') or '').lower()==t and r.get('source_schema'):
            return r['source_schema']
        if str(r.get('sub_target_table','') or '').lower()==t and r.get('sub_target_schema'):
            return r['sub_target_schema']
    return ''

def trace(col, tbl):
    nf = lambda a,b: f'{a.lower()}::{b.lower()}'
    nodes, nrows, edges, visited = {}, {}, {}, set()
    q = deque()

    def add(t, c, sc, start=False):
        k = nf(t, c)
        if k not in nodes:
            nodes[k] = {'id':k,'tbl':t,'col':c,'schema':sc,'is_start':start}
            nrows[k] = []
        return k

    seed = [r for r in _rows if is_profile(r)
            and str(r.get('target_table',  '') or '').lower()==tbl.lower()
            and str(r.get('target_column', '') or '').lower()==col.lower()]
    if not seed:
        return None

    sid = add(tbl, col, seed[0].get('target_schema',''), True)
    nrows[sid] = seed
    visited.add(sid)
    for r in seed:
        q.append((r.get('source_table','') or '', r.get('source_column','') or '', sid, [r]))

    itr = 0
    while q and itr < 800:
        itr += 1
        stbl, scol, pid, tr = q.popleft()

        if not stbl and scol:
            up = [r for r in _rows if is_profile(r)
                  and str(r.get('target_column','') or '').lower()==scol.lower()]
            for r in up:
                uid = add(r['target_table'], scol, r.get('target_schema',''))
                ek  = f'{uid}>{pid}'
                if ek not in edges: edges[ek]={'from':uid,'to':pid}
                nrows[uid].append(r)
                if uid not in visited:
                    visited.add(uid)
                    sub = _get(r['target_table'], scol)
                    nrows[uid].extend(sub)
                    for sr in sub:
                        q.append((sr.get('source_table','') or '',
                                  sr.get('source_column','') or '', uid, [sr]))
            continue

        if not stbl and not scol:
            continue

        sv   = _schema(stbl)
        sid2 = add(stbl, scol, sv)
        ek   = f'{sid2}>{pid}'
        if ek not in edges: edges[ek]={'from':sid2,'to':pid}
        if sid2 in visited: continue
        visited.add(sid2)
        sub = _get(stbl, scol)
        nrows[sid2].extend(sub)
        for sr in sub:
            q.append((sr.get('source_table','') or '',
                      sr.get('source_column','') or '', sid2, [sr]))

    return {'nodes':list(nodes.values()), 'edges':list(edges.values()), 'nrows':nrows}

SAFE = ['target_table','target_schema','sub_target_table','sub_target_schema',
        'target_column','source_table','source_schema','source_column',
        'process','sql_process','logic']

def clean_nrows(nrows):
    out = {}
    for k, rv in nrows.items():
        seen_set, ded = set(), []
        for r in rv:
            rk = (r.get('sql_process',''), r.get('source_table',''),
                  r.get('source_column',''), r.get('target_column',''))
            if rk not in seen_set:
                seen_set.add(rk)
                ded.append({f: str(r.get(f,'') or '') for f in SAFE})
        out[k] = ded
    return out

def safe_name(s): return re.sub(r'[^a-zA-Z0-9_-]', '_', str(s).strip())

def make_html(graphs, title, subtitle, ts_str):
    # graphs = list of dicts with keys: profile_table, rule_column, nodes, edges, nrows
    gd_json = json.dumps(graphs, ensure_ascii=False)
    return (HTML_TEMPLATE
            .replace('__TITLE__',      title)
            .replace('__SUBTITLE__',   subtitle)
            .replace('__TS__',         ts_str)
            .replace('__ALL_GRAPHS__', gd_json))

print('Helpers ready')


## 6. HTML Template

In [ ]:
# HTML template with vis-network (rectangular box nodes, hierarchical LR layout)
HTML_TEMPLATE = '<!DOCTYPE html>\n<html lang="en"><head><meta charset="UTF-8">\n<title>__TITLE__</title>\n<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/dist/vis-network.min.js"></script>\n<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/dist/dist/vis-network.min.css">\n<style>\n*{box-sizing:border-box;margin:0;padding:0;}\nbody{font-family:Inter,Segoe UI,Arial,sans-serif;height:100vh;display:flex;flex-direction:column;overflow:hidden;background:#f4f7fb;}\n#bar{background:linear-gradient(120deg,#0d1f3c,#1a3a6e 55%,#1565c0);color:#fff;padding:10px 22px;display:flex;align-items:center;gap:14px;box-shadow:0 2px 12px rgba(0,0,0,.32);flex-shrink:0;}\n#bar h1{font-size:16px;font-weight:800;white-space:nowrap;}\n#bar h1 em{color:#90caf9;font-style:normal;}\n.bsep{width:1px;height:24px;background:rgba(255,255,255,.22);flex-shrink:0;}\n.bmeta{font-size:12px;color:rgba(255,255,255,.62);}\n.bmeta b{color:rgba(255,255,255,.92);}\n#bts{font-size:11px;color:rgba(255,255,255,.38);margin-left:auto;}\n#wrap{display:flex;flex:1;overflow:hidden;}\n#graphs{flex:1;overflow-y:auto;display:flex;flex-direction:column;}\n.gsec{border-bottom:3px solid #d0dae8;flex-shrink:0;}\n.ghdr{background:linear-gradient(90deg,#1a3a6e,#1565c0);color:#fff;padding:9px 18px;display:flex;align-items:center;gap:10px;font-size:13px;flex-wrap:wrap;}\n.ghdr .tbl{background:rgba(255,255,255,.15);border-radius:6px;padding:2px 10px;font-size:12px;}\n.ghdr .col{background:rgba(255,255,255,.25);border-radius:6px;padding:2px 10px;font-family:monospace;font-size:12px;font-weight:700;}\n.ghdr .cnt{margin-left:auto;font-size:11px;color:rgba(255,255,255,.55);}\n.gwrap{position:relative;height:520px;background:linear-gradient(150deg,#eef2fa,#e6edf8);}\n.gnet{width:100%;height:100%;}\n.gleg{position:absolute;bottom:10px;left:10px;background:rgba(255,255,255,.94);border:1px solid #c8d8ee;border-radius:9px;padding:9px 13px;max-width:220px;box-shadow:0 2px 10px rgba(0,0,0,.1);}\n.gleg h5{font-size:9px;font-weight:800;text-transform:uppercase;letter-spacing:.6px;color:#4a6080;margin-bottom:6px;}\n.lrow{display:flex;align-items:center;gap:7px;margin-bottom:4px;font-size:11px;color:#263248;}\n.ldot{width:13px;height:13px;border-radius:3px;flex-shrink:0;border:1.5px solid rgba(0,0,0,.18);}\n.gcam{position:absolute;bottom:10px;right:10px;display:flex;flex-direction:column;gap:5px;}\n.cbtn{width:30px;height:30px;background:#fff;border:1.5px solid #b8cde8;color:#1a3a6e;border-radius:7px;cursor:pointer;font-size:16px;font-weight:800;display:flex;align-items:center;justify-content:center;box-shadow:0 2px 6px rgba(0,0,0,.12);transition:all .15s;}\n.cbtn:hover{background:#1565c0;color:#fff;border-color:#1565c0;}\n#rp{width:330px;background:#fff;border-left:1.5px solid #d0dae8;display:flex;flex-direction:column;flex-shrink:0;overflow:hidden;}\n#rphdr{background:linear-gradient(90deg,#0d1f3c,#1a3a6e);color:#fff;padding:10px 16px;font-size:13px;font-weight:700;flex-shrink:0;}\n#rpbody{flex:1;overflow-y:auto;padding:12px;}\n.rphint{display:flex;flex-direction:column;align-items:center;justify-content:center;height:100%;gap:10px;color:#7a92b0;text-align:center;padding:24px;}\n.rphint .ico{font-size:40px;opacity:.4;}\n.rphint p{font-size:13px;line-height:1.6;}\n.ncard{margin-bottom:9px;border:1.5px solid #d0dae8;border-radius:9px;overflow:hidden;box-shadow:0 1px 5px rgba(0,0,0,.05);}\n.nhdr{padding:9px 12px;}\n.ntbl{font-size:14px;font-weight:800;color:#fff;letter-spacing:.15px;display:block;}\n.ncol{font-size:11px;color:rgba(255,255,255,.72);font-weight:500;font-style:italic;display:block;margin-top:2px;}\n.nrow{display:flex;padding:5px 12px;border-bottom:1px solid #eef2f9;font-size:13px;}\n.nrow:last-child{border-bottom:none;}\n.nlbl{color:#607090;width:118px;flex-shrink:0;font-size:11.5px;font-weight:600;}\n.nval{color:#1a2744;word-break:break-word;font-weight:500;}\n.nval.empty{color:#b0bec8;font-style:italic;font-weight:400;}\n.nval.mono{font-family:monospace;font-size:11px;background:#f2f6fc;padding:2px 5px;border-radius:4px;color:#1a3a6e;}\n.rb{display:inline-block;padding:2px 8px;border-radius:8px;font-size:11px;font-weight:700;}\n.rb-s{background:#dbeafe;color:#1d4ed8;}.rb-j{background:#d1fae5;color:#065f46;}\n.rb-w{background:#fef3c7;color:#92400e;}.rb-h{background:#fce7f3;color:#9d174d;}\n.rb-v{background:#ede9fe;color:#5b21b6;}.rb-x{background:#cffafe;color:#155e75;}\n#rpbody::-webkit-scrollbar,#graphs::-webkit-scrollbar{width:5px;}\n#rpbody::-webkit-scrollbar-thumb,#graphs::-webkit-scrollbar-thumb{background:#c0cfe8;border-radius:3px;}\n</style></head><body>\n<div id="bar">\n  <h1>IKG Lineage <em>Explorer</em></h1>\n  <div class="bsep"></div>\n  <div class="bmeta">__SUBTITLE__</div>\n  <div id="bts">__TS__</div>\n</div>\n<div id="wrap">\n  <div id="graphs"></div>\n  <div id="rp">\n    <div id="rphdr">&#128202; Node Details</div>\n    <div id="rpbody"><div class="rphint"><div class="ico">&#128269;</div><p>Click any node to see<br>lineage details here.</p></div></div>\n  </div>\n</div>\n<script>\nvar ALL_GRAPHS  = __ALL_GRAPHS__;\nvar RAW2COLOR   = {"ikg_schema": "#1565c0", "sandbox_prj_smart_insights": "#1565c0", "sandbox_ikg_pre_prd": "#1565c0", "edw_input_schema": "#00796b", "edw_view_input_schema": "#00796b", "ikg_vendor_schema": "#00796b", "core_wma_shared": "#00796b", "sandbox_wma_shared": "#2e7d32", "core_wma_shared_masked": "#00838f", "sandbox_prj_ds_data": "#e65100", "sandbox_prj_sbl": "#6a1b9a", "core_nlg": "#4527a0", "nlg_schema": "#4527a0", "core_model": "#4e342e", "model_schema": "#4e342e", "sandbox_prj_dsforoverdrive": "#0277bd", "core_in_shared": "#558b2f", "ikg_clip_schema": "#558b2f", "sandbox_prj_adhoc": "#ad1457", "ikg_wealthx_schema": "#006064", "sandbox_prj_rbat": "#827717", "_default": "#455a64"};\nvar LEGEND_DATA = [{"label": "core_ikg", "color": "#1565c0"}, {"label": "core_wma_shared", "color": "#00796b"}, {"label": "sandbox_wma_shared", "color": "#2e7d32"}, {"label": "core_wma_shared_masked", "color": "#00838f"}, {"label": "sandbox_prj_ds_data", "color": "#e65100"}, {"label": "sandbox_prj_sbl", "color": "#6a1b9a"}, {"label": "core_nlg", "color": "#4527a0"}, {"label": "core_model", "color": "#4e342e"}, {"label": "sandbox_prj_dsforoverdrive", "color": "#0277bd"}, {"label": "core_in_shared", "color": "#558b2f"}, {"label": "sandbox_prj_adhoc", "color": "#ad1457"}, {"label": "sandbox_prj_smart_relationship", "color": "#006064"}, {"label": "sandbox_prj_rbat", "color": "#827717"}];\n\nfunction getColor(schema){\n  if(!schema) return RAW2COLOR._default;\n  var s = schema.toLowerCase().trim();\n  if(RAW2COLOR[s]) return RAW2COLOR[s];\n  for(var k in RAW2COLOR){\n    if(k === "_default") continue;\n    if(s.indexOf(k) !== -1 || k.indexOf(s) !== -1) return RAW2COLOR[k];\n  }\n  return RAW2COLOR._default;\n}\n\nfunction buildLegend(nodes){\n  var seen = {};\n  nodes.forEach(function(n){ seen[getColor(n.schema)] = true; });\n  var html = "<h5>Schema</h5>";\n  LEGEND_DATA.forEach(function(e){\n    if(seen[e.color]){\n      html += "<div class=\\"lrow\\"><div class=\\"ldot\\" style=\\"background:" + e.color + "\\"></div><span>" + esc(e.label) + "</span></div>";\n    }\n  });\n  return html;\n}\n\nvar nets = [];\n\nwindow.onload = function(){ buildAll(); };\n\nfunction buildAll(){\n  var col = document.getElementById("graphs");\n  col.innerHTML = "";\n  nets = [];\n  ALL_GRAPHS.forEach(function(g, gi){ buildSection(g, gi, col); });\n}\n\nfunction buildSection(g, gi, col){\n  var sec  = document.createElement("div"); sec.className = "gsec";\n  var hdr  = document.createElement("div"); hdr.className = "ghdr";\n  hdr.innerHTML =\n    "<span class=\\"tbl\\">&#128203; " + esc(g.profile_table) + "</span>"\n    + "<span class=\\"col\\">" + esc(g.rule_column) + "</span>"\n    + "<span class=\\"cnt\\">" + g.nodes.length + " nodes &middot; " + g.edges.length + " edges</span>";\n  sec.appendChild(hdr);\n\n  var gw   = document.createElement("div"); gw.className = "gwrap";\n  var gnet = document.createElement("div"); gnet.className = "gnet"; gnet.id = "gn" + gi;\n  gw.appendChild(gnet);\n\n  var leg  = document.createElement("div"); leg.className = "gleg";\n  leg.innerHTML = buildLegend(g.nodes);\n  gw.appendChild(leg);\n\n  var cam  = document.createElement("div"); cam.className = "gcam";\n  cam.innerHTML =\n    "<button class=\\"cbtn\\" onclick=\\"nz(" + gi + ",1.3)\\">+</button>"\n    + "<button class=\\"cbtn\\" onclick=\\"nz(" + gi + ",0.77)\\">&#8722;</button>"\n    + "<button class=\\"cbtn\\" onclick=\\"nf(" + gi + ")\\">&#8861;</button>";\n  gw.appendChild(cam);\n  sec.appendChild(gw);\n  col.appendChild(sec);\n\n  var vNodes = new vis.DataSet();\n  var vEdges = new vis.DataSet();\n\n  g.nodes.forEach(function(n){\n    var bg  = getColor(n.schema);\n    var bdr = shadeColor(bg, -30);\n    vNodes.add({\n      id:    n.id,\n      shape: "box",\n      label: n.tbl + "\\n" + n.col,\n      font: {\n        multi:    "md",\n        size:     13,\n        face:     "Inter, Segoe UI, Arial",\n        color:    "#ffffff",\n        bold:     { size: 14, color: "#ffffff", vadjust: 0 },\n        ital:     { size: 11, color: "rgba(255,255,255,0.78)", vadjust: 0 }\n      },\n      label: "**" + n.tbl + "**\\n_" + n.col + "_",\n      widthConstraint:  { minimum: 130, maximum: 210 },\n      heightConstraint: { minimum: 44 },\n      margin: 10,\n      color: {\n        background: bg,\n        border:     bdr,\n        highlight:  { background: shadeColor(bg,20), border: bdr },\n        hover:      { background: shadeColor(bg,20), border: bdr }\n      },\n      borderWidth:         n.is_start ? 3 : 1.5,\n      borderWidthSelected: 4,\n      shadow: { enabled: true, size: 6, x: 2, y: 2, color: "rgba(0,0,0,0.18)" }\n    });\n  });\n\n  g.edges.forEach(function(e, i){\n    vEdges.add({\n      id:     "e" + gi + "_" + i,\n      from:   e.from,\n      to:     e.to,\n      arrows: { to: { enabled: true, scaleFactor: 0.85 } },\n      color:  { color: "#5c7ec0", highlight: "#1a3a6e", hover: "#1565c0", opacity: 0.85 },\n      width:  2,\n      smooth: { type: "cubicBezier", forceDirection: "horizontal", roundness: 0.4 }\n    });\n  });\n\n  var net = new vis.Network(gnet,\n    { nodes: vNodes, edges: vEdges },\n    {\n      layout: {\n        hierarchical: {\n          enabled:              true,\n          direction:            "LR",\n          sortMethod:           "directed",\n          nodeSpacing:          90,\n          levelSeparation:      230,\n          treeSpacing:          160,\n          blockShifting:        true,\n          edgeMinimization:     true,\n          parentCentralization: true\n        }\n      },\n      physics: { enabled: false },\n      interaction: {\n        hover:        true,\n        tooltipDelay: 150,\n        zoomView:     true,\n        dragView:     true\n      },\n      nodes: { shapeProperties: { borderRadius: 7 } },\n      edges: { selectionWidth: 3 }\n    }\n  );\n\n  net.on("click", function(p){\n    if(p.nodes.length > 0) showDetail(g, p.nodes[0]);\n  });\n  nets.push(net);\n  setTimeout(function(){ net.fit({ animation: { duration: 450, easingFunction: "easeInOutQuad" } }); }, 220);\n}\n\nfunction nz(gi, f){ if(nets[gi]){ var s=nets[gi].getScale()*f; nets[gi].moveTo({scale:s,animation:{duration:200}}); } }\nfunction nf(gi){ if(nets[gi]) nets[gi].fit({animation:{duration:350}}); }\n\nfunction shadeColor(hex, pct){\n  try{\n    var r=parseInt(hex.slice(1,3),16),g=parseInt(hex.slice(3,5),16),b=parseInt(hex.slice(5,7),16);\n    r=Math.min(255,Math.max(0,r+Math.round(r*pct/100)));\n    g=Math.min(255,Math.max(0,g+Math.round(g*pct/100)));\n    b=Math.min(255,Math.max(0,b+Math.round(b*pct/100)));\n    return "#"+("0"+r.toString(16)).slice(-2)+("0"+g.toString(16)).slice(-2)+("0"+b.toString(16)).slice(-2);\n  }catch(e){ return hex; }\n}\n\nfunction showDetail(g, nodeId){\n  var nd=null; g.nodes.forEach(function(n){ if(n.id===nodeId) nd=n; });\n  if(!nd) return;\n  var rows = g.nrows[nodeId]||[];\n  var bg   = getColor(nd.schema);\n  var bdr  = shadeColor(bg,-30);\n  var body = document.getElementById("rpbody");\n  var seen = new Set(), ded=[];\n  rows.forEach(function(r){\n    var k=r.sql_process+"|"+r.source_table+"|"+r.source_column+"|"+r.target_column;\n    if(!seen.has(k)){ seen.add(k); ded.push(r); }\n  });\n  var h = "<div class=\\"ncard\\">"\n    + "<div class=\\"nhdr\\" style=\\"background:"+bg+";border-bottom:3px solid "+bdr+"\\">"\n    + "<span class=\\"ntbl\\">&#128200; "+esc(nd.tbl)+"</span>"\n    + "<span class=\\"ncol\\">col: "+esc(nd.col)+"</span>"\n    + "</div><div>"\n    + nr("Schema",  nd.schema)\n    + nr("Role",    nd.is_start ? "&#9733; Profile Target" : "Source / Intermediate")\n    + "</div></div>";\n  if(!ded.length){\n    h += "<div style=\\"color:#7a92b0;font-size:13px;padding:16px;text-align:center;line-height:1.7\\">"\n      + "<div style=\\"font-size:28px;margin-bottom:6px\\">&#128204;</div>"\n      + "Base source &#8212; no further lineage.</div>";\n    body.innerHTML=h; return;\n  }\n  ded.forEach(function(r,i){\n    var p=r.sql_process||"select";\n    h += "<div class=\\"ncard\\">"\n      + "<div class=\\"nhdr\\" style=\\"background:"+bg+"22;border-bottom:2px solid "+bdr+"44\\">"\n      + bdg(p)+"<span style=\\"font-size:11px;color:#4a6080;font-weight:600;margin-left:5px\\">"+(i+1)+"/"+ded.length+"</span>"\n      + "</div><div>"\n      + nr("Target Table",     r.target_table)\n      + nr("Target Schema",    r.target_schema)\n      + nr("Sub-Target Table", r.sub_target_table)\n      + nr("Sub-Target Schema",r.sub_target_schema)\n      + nr("Target Column",    r.target_column)\n      + nr("Source Table",     r.source_table)\n      + nr("Source Schema",    r.source_schema)\n      + nr("Source Column",    r.source_column)\n      + nr("Process",          r.process)\n      + nr("SQL Process",      r.sql_process)\n      + (r.logic ? nrm("Logic",r.logic.slice(0,450)) : "")\n      + "</div></div>";\n  });\n  body.innerHTML=h;\n}\n\nfunction nr(l,v){ var e=!v||!String(v).trim(); return "<div class=\\"nrow\\"><span class=\\"nlbl\\">"+l+"</span><span class=\\"nval"+(e?" empty":"")+"\\"> "+(e?"-":esc(String(v)))+"</span></div>"; }\nfunction nrm(l,v){ return "<div class=\\"nrow\\"><span class=\\"nlbl\\">"+l+"</span><span class=\\"nval mono\\">"+esc(String(v))+"</span></div>"; }\nfunction bdg(p){\n  var m={"select":"rb-s","select-value":"rb-v","select*":"rb-x","join":"rb-j","where":"rb-w","having":"rb-h","where-subquery":"rb-w"};\n  return "<span class=\\"rb "+(m[p]||"rb-s")+"\\">"+esc(p||"select")+"</span>";\n}\nfunction esc(s){ return String(s||"").replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/"/g,"&quot;"); }\n</script></body></html>'
print(f'Template: {len(HTML_TEMPLATE):,} chars')


## Mode 1 — Column / Table Lineage

1. Type to search a **column** in the dropdown (blank values excluded)
2. **Profile tables** auto-populate in the second dropdown
3. Click **▶ Trace** → saves `<table>_<column>_lineage.html`


In [ ]:
_s1 = widgets.Output()

w_col = widgets.Combobox(
    options=COLS_LIST, value='',
    placeholder='Type to search column...',
    description='Column:', ensure_option=True,
    style={'description_width':'80px'},
    layout=widgets.Layout(width='400px')
)
w_tbl = widgets.Combobox(
    options=[], value='',
    placeholder='Select a column first...',
    description='Table:', ensure_option=True,
    style={'description_width':'80px'},
    layout=widgets.Layout(width='400px'), disabled=True
)
btn1 = widgets.Button(
    description='▶  Trace', button_style='primary', disabled=True,
    layout=widgets.Layout(width='120px', height='36px')
)

def _on_col(change):
    col = (change['new'] or '').strip()
    with _s1: clear_output()
    w_tbl.value = ''; btn1.disabled = True
    if not col:
        w_tbl.options = []; w_tbl.disabled = True; return
    tbls = sorted(df_profile[df_profile['target_column']==col]['target_table']
                  .dropna().unique().tolist())
    w_tbl.options = tbls; w_tbl.disabled = False
    w_tbl.placeholder = f'{len(tbls)} table(s) — type to filter...'
    with _s1: print(f'Found {len(tbls)} profile table(s) for "{col}"')

def _on_tbl(change):
    btn1.disabled = not bool((change['new'] or '').strip())

w_col.observe(_on_col, names='value')
w_tbl.observe(_on_tbl, names='value')

def _trace1(_):
    col = (w_col.value or '').strip()
    tbl = (w_tbl.value or '').strip()
    if not col or not tbl: return
    with _s1: clear_output(); print(f'Tracing {tbl}.{col} ...')
    res = trace(col, tbl)
    if not res:
        with _s1: clear_output(); print(f'No lineage found for {tbl}.{col}')
        return
    graph = {'profile_table': tbl, 'rule_column': col,
             'nodes': res['nodes'], 'edges': res['edges'],
             'nrows': clean_nrows(res['nrows'])}
    ts    = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    sub   = f'Table: <b>{tbl}</b>&nbsp;&nbsp;Column: <b>{col}</b>'
    html  = make_html([graph], f'{tbl}.{col}', sub, ts)
    fname = safe_name(tbl) + '_' + safe_name(col) + '_lineage.html'
    fpath = os.path.abspath(fname)
    with open(fname, 'w', encoding='utf-8') as f: f.write(html)
    with _s1:
        clear_output()
        print(f'{len(res["nodes"])} nodes, {len(res["edges"])} edges')
        print(f'Saved: {fpath}')
        print('Open in Chrome / Edge / Firefox.')

btn1.on_click(_trace1)
ipy_display(widgets.VBox([widgets.HBox([w_col, w_tbl, btn1]), _s1]))


## Mode 2 — Insight Type Lineage

1. Type to search an **insight type** in the dropdown
2. Preview table shows all `(profile_table, rule_column)` pairs
3. Click **▶ Trace Lineage** → saves `<insight_type>_lineage.html`
   with **one graph section per rule_column** (titled with profile_table + rule_column)


In [ ]:
_s2   = widgets.Output()
_tbl2 = widgets.Output()
_scope2 = {'df': None}

w_ins = widgets.Combobox(
    options=IT_LIST, value='',
    placeholder='Type to search insight type...',
    description='Insight Type:', ensure_option=True,
    style={'description_width':'105px'},
    layout=widgets.Layout(width='500px')
)
btn2 = widgets.Button(
    description='▶  Trace Lineage', button_style='primary', disabled=True,
    layout=widgets.Layout(width='160px', height='36px')
)

def _on_ins(change):
    sel = (change['new'] or '').strip()
    btn2.disabled = True
    _scope2['df'] = None
    with _tbl2: clear_output()
    with _s2:   clear_output()
    if not sel: return
    with _s2: print(f'Loading metadata for: {sel} ...')
    try:
        q = ('SELECT DISTINCT profile_table, rule_column FROM '
             + ODM_SCHEMA + '.' + ODM_TABLE
             + " WHERE insight_type = %(s)s"
             + " AND profile_table IS NOT NULL AND profile_table <> ''"
             + " AND rule_column   IS NOT NULL AND rule_column   <> ''")
        df_s = pd.read_sql(q, engine, params={'s': sel})
        if df_s.empty:
            with _s2: clear_output(); print(f'No metadata found for: {sel}')
            return
        _scope2['df'] = df_s
        btn2.disabled = False
        with _s2: clear_output(); print(f'{len(df_s)} pair(s) found — click Trace Lineage')
        with _tbl2:
            clear_output()
            ipy_display(IPHTML(
                '<b>profile_table &amp; rule_column for insight: </b><code>' + sel + '</code><br><br>'
                + df_s.to_html(index=False, border=0, justify='left')
            ))
    except Exception as e:
        with _s2: clear_output(); print(f'Error: {e}')

w_ins.observe(_on_ins, names='value')

def _trace2(_):
    sel      = (w_ins.value or '').strip()
    df_scope = _scope2.get('df')
    if not sel or df_scope is None: return
    with _s2: clear_output(); print(f'Tracing {len(df_scope)} pair(s)...')

    # One separate graph per (profile_table, rule_column) pair
    graphs = []
    for _, row in df_scope.iterrows():
        ptbl = str(row['profile_table']).strip()
        rcol = str(row['rule_column']).strip()
        res  = trace(rcol, ptbl)
        if not res: continue
        graphs.append({
            'profile_table': ptbl,
            'rule_column':   rcol,
            'nodes':  res['nodes'],
            'edges':  res['edges'],
            'nrows':  clean_nrows(res['nrows']),
        })

    if not graphs:
        with _s2: clear_output(); print('No lineage found for any pair.')
        return

    ts    = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    sub   = f'Insight: <b>{sel}</b>&nbsp;&nbsp;Sections: <b>{len(graphs)}</b>'
    html  = make_html(graphs, sel, sub, ts)
    fname = safe_name(sel) + '_lineage.html'
    fpath = os.path.abspath(fname)
    with open(fname, 'w', encoding='utf-8') as f: f.write(html)
    tn = sum(len(g['nodes']) for g in graphs)
    te = sum(len(g['edges']) for g in graphs)
    with _s2:
        clear_output()
        print(f'{len(graphs)} section(s), {tn} total nodes, {te} total edges')
        print(f'Saved: {fpath}')
        print('Open in Chrome / Edge / Firefox.')

btn2.on_click(_trace2)
ipy_display(widgets.VBox([
    widgets.HBox([w_ins, btn2]),
    _tbl2,
    _s2,
]))


## Usage Guide

### Mode 1 — Column / Table
| Step | Action |
|------|--------|
| 1 | Run cells 1–5 |
| 2 | Type to search **column** (blanks/nulls excluded) |
| 3 | Profile tables auto-populate — pick one |
| 4 | Click **▶ Trace** → saves `<table>_<column>_lineage.html` |

### Mode 2 — Insight Type
| Step | Action |
|------|--------|
| 1 | Type to search **insight type** |
| 2 | Preview table shows all `(profile_table, rule_column)` pairs |
| 3 | Click **▶ Trace Lineage** → saves `<insight_type>_lineage.html` |

### HTML Graph Features
- **Rectangular boxes** — table name **bold**, column name *italic* below
- **Left → Right** layout — source tables on the left, profile target on the right
- **No overlapping** — vis-network hierarchical engine with auto-spacing
- **Visible edges** — curved arrows with arrowheads (width 2px, blue)
- **Separate section per rule_column** (Mode 2) — each titled with profile_table + rule_column
- **+/−/⊡** camera controls per graph section
- **Click any node** → right panel shows all 11 lineage fields
- **Legend per section** shows only schemas present in that graph

### Schema → Canonical Group → Colour
| Canonical Group | Raw schemas | Colour |
|----------------|-------------|--------|
| `core_ikg` | `IKG_SCHEMA`, `sandbox_prj_smart_insights`, `sandbox_ikg_pre_prd` | Deep Blue |
| `core_wma_shared` | `EDW_INPUT_SCHEMA`, `EDW_VIEW_INPUT_SCHEMA`, `IKG_VENDOR_SCHEMA`, `core_wma_shared` | Teal |
| `core_nlg` | `NLG_SCHEMA`, `core_nlg` | Deep Purple |
| `core_model` | `MODEL_SCHEMA`, `core_model` | Brown |
| `core_in_shared` | `IKG_CLIP_SCHEMA`, `core_in_shared` | Lime Green |
| `sandbox_prj_smart_relationship` | `IKG_WEALTHX_SCHEMA` | Dark Cyan |
